# HaulMark EDA
This notebook runs exploratory data checks and saves plots to eda/.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from utils.data_loader import discover_data_files, split_train_test_files, load_telemetry
from utils.preprocessing import clean_telemetry
Path('eda').mkdir(exist_ok=True)
files = discover_data_files(Path('data'))
train_files, test_files = split_train_test_files(files.telemetry_files)
print('Train files:', [f.name for f in train_files])
print('Test files:', [f.name for f in test_files])

In [ ]:
cols = ['vehicle','ts','date_dpr','latitude','longitude','altitude','speed','ignition','analog_input_1','satellites','gnss_hdop','disthav','fuel_volume','total_trip']
df = load_telemetry(train_files[0], columns=cols)
df = clean_telemetry(df)
print(df.shape)
print(df.dtypes)
print(df.isnull().sum().sort_values(ascending=False).head(20))
print('Unique vehicles', df['vehicle'].nunique())
print('Date range', df['ts'].min(), df['ts'].max())

In [ ]:
daily = df.groupby(['vehicle','date_key'], observed=True).agg(fuel=('fuel_volume','first'), speed_mean=('speed','mean'), rows=('ts','count')).reset_index()
plt.figure(figsize=(8,4))
sns.histplot(daily['fuel'].dropna(), bins=50)
plt.title('Daily Fuel Distribution')
plt.tight_layout()
plt.savefig('eda/daily_fuel_hist.png', dpi=150)
plt.close()
plt.figure(figsize=(10,4))
sns.boxplot(data=daily, x='vehicle', y='fuel')
plt.xticks(rotation=90)
plt.tight_layout()
plt.savefig('eda/vehicle_fuel_boxplot.png', dpi=150)
plt.close()